# 14 — KASBA Clustering

**KASBA** (K-means with Accelerated Stochastic Barycenter Averaging) is a time series
clustering algorithm that uses the **MSM** (Move-Split-Merge) elastic distance instead
of DTW.

### Why MSM over DTW?

| Property | DTW | MSM |
|----------|-----|-----|
| **Metric** | No (violates triangle inequality) | Yes (proper metric) |
| **Triangle inequality pruning** | Not possible | 10-30x speedup |
| **Phase invariance** | Yes | Yes |
| **Amplitude invariance** | No | Tunable via cost `c` |

Because MSM is a proper metric, KASBA can use triangle inequality pruning to skip
unnecessary distance computations — making it significantly faster than DTW-based
k-means on large datasets.

### How KASBA works

1. Initialize `k` centroids (random selection from data)
2. **Assign** each series to nearest centroid using MSM distance
3. **Update** centroids via stochastic barycenter averaging (SGD on a subset of cluster members)
4. Repeat until convergence or `max_iter` reached

The stochastic barycenter is cheaper than the full DBA (Dynamic Barycenter Averaging)
used in DTW k-means, while still producing high-quality centroids.

### References

- Stefan, A., Athitsos, V. & Das, G. (2013). *The Move-Split-Merge Metric for Time Series*. IEEE TKDE.
- Holder, C., Middlehurst, M. & Bagnall, A. (2023). *A Review and Evaluation of Elastic Distance Functions for Time Series Clustering*. Knowledge and Information Systems.

In [ ]:
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]
if importlib.util.find_spec("hvplot") is None:
    %pip install -q hvplot

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass

import numpy as np
import polars as pl

from polars_ts import KASBAClusterer, TimeSeriesKMeans, kasba, silhouette_score

## 14.1 Synthetic data

We generate 30 univariate series belonging to 3 shape families:
- **Sine** — smooth periodic pattern
- **Sawtooth** — linear ramp with reset
- **Step** — piecewise constant with a level shift

Each family has 10 series with slight noise, making them visually distinct but
requiring an elastic distance to cluster correctly.

In [ ]:
rng = np.random.default_rng(42)
n_timepoints = 50
n_per_cluster = 10

rows = []
ground_truth = {}

for i in range(n_per_cluster):
    # Sine family
    vals = np.sin(np.linspace(0, 4 * np.pi, n_timepoints)) + rng.normal(0, 0.15, n_timepoints)
    uid = f"sine_{i}"
    ground_truth[uid] = 0
    for t, v in enumerate(vals):
        rows.append({"unique_id": uid, "ds": t, "y": v})

    # Sawtooth family
    vals = np.linspace(0, 1, n_timepoints) + rng.normal(0, 0.1, n_timepoints)
    uid = f"saw_{i}"
    ground_truth[uid] = 1
    for t, v in enumerate(vals):
        rows.append({"unique_id": uid, "ds": t, "y": v})

    # Step family
    vals = np.concatenate([np.zeros(25), np.ones(25)]) + rng.normal(0, 0.1, n_timepoints)
    uid = f"step_{i}"
    ground_truth[uid] = 2
    for t, v in enumerate(vals):
        rows.append({"unique_id": uid, "ds": t, "y": v})

df = pl.DataFrame(rows)
print(f"Shape: {df.shape}")
print(f"Series: {df['unique_id'].n_unique()}")
df.head()

In [ ]:
df.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    width=900,
    height=400,
    title="30 Synthetic Series (3 Shape Families)",
    alpha=0.5,
    legend=False,
)

## 14.2 Univariate clustering with `kasba()`

The simplest way to use KASBA — one function call. Returns a DataFrame
with `[unique_id, cluster]` assignments.

In [ ]:
labels = kasba(df, k=3, seed=42)
print("KASBA cluster assignments:")
labels

In [ ]:
# Visualize clusters
df_clustered = df.join(labels, on="unique_id")

df_clustered.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    groupby="cluster",
    width=900,
    height=400,
    title="KASBA Clusters (k=3, MSM c=1.0)",
    alpha=0.7,
    legend=False,
)

In [ ]:
# Check agreement with ground truth
gt_df = pl.DataFrame({"unique_id": list(ground_truth.keys()), "true_cluster": list(ground_truth.values())})
comparison = labels.join(gt_df, on="unique_id")

# Cluster IDs may differ from ground truth IDs, so check purity
for cluster_id in comparison["cluster"].unique().sort().to_list():
    members = comparison.filter(pl.col("cluster") == cluster_id)
    true_labels = members["true_cluster"].value_counts().sort("count", descending=True)
    purity = true_labels["count"][0] / members.height
    print(f"Cluster {cluster_id}: {members.height} members, purity = {purity:.0%}")

## 14.3 Multivariate clustering

KASBA supports multivariate time series via a `channel_col` parameter. Each series
has multiple channels (e.g., temperature and humidity), and MSM is computed either:

- **independently** (`independent=True`, default) — sum of per-channel MSM distances
- **dependently** (`independent=False`) — cross-channel MSM that considers inter-channel relationships

In [ ]:
# Create multivariate data: 12 series, 2 channels, 2 clusters
rng_mv = np.random.default_rng(99)
mv_rows = []
for i in range(6):
    for ch in ["temp", "humidity"]:
        vals = rng_mv.normal(0, 1, 30)
        for t, v in enumerate(vals):
            mv_rows.append({"unique_id": f"cold_{i}", "ds": t, "channel": ch, "y": v})
for i in range(6):
    for ch in ["temp", "humidity"]:
        vals = rng_mv.normal(8, 1, 30)
        for t, v in enumerate(vals):
            mv_rows.append({"unique_id": f"hot_{i}", "ds": t, "channel": ch, "y": v})

df_mv = pl.DataFrame(mv_rows)
print(f"Shape: {df_mv.shape}, Series: {df_mv['unique_id'].n_unique()}, Channels: {df_mv['channel'].n_unique()}")

In [ ]:
# Independent mode (default)
clf_ind = KASBAClusterer(n_clusters=2, independent=True, seed=42)
clf_ind.fit(df_mv, channel_col="channel")
print("Independent mode:")
print(clf_ind.labels_)
print(f"Inertia: {clf_ind.inertia_:.2f}, Iterations: {clf_ind.n_iter_}")

# Dependent mode
clf_dep = KASBAClusterer(n_clusters=2, independent=False, seed=42)
clf_dep.fit(df_mv, channel_col="channel")
print("\nDependent mode:")
print(clf_dep.labels_)
print(f"Inertia: {clf_dep.inertia_:.2f}, Iterations: {clf_dep.n_iter_}")

## 14.4 Tuning MSM cost `c`

The MSM cost parameter `c` controls the penalty for split and merge operations:

- **Low `c`** (e.g., 0.1) — cheap to split/merge, tolerates amplitude differences → fewer, broader clusters
- **High `c`** (e.g., 10.0) — expensive to split/merge, sensitive to amplitude → tighter, more clusters needed

This is analogous to the Sakoe-Chiba band width in DTW — it controls how much
deformation the distance allows.

In [ ]:
# Sweep c values and measure silhouette score
results = []
for c_val in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    lbl = kasba(df, k=3, c=c_val, seed=42)
    sil = silhouette_score(df, lbl, method="msm")
    clf_tmp = KASBAClusterer(n_clusters=3, c=c_val, seed=42)
    clf_tmp.fit(df)
    results.append({"c": c_val, "silhouette": sil, "inertia": clf_tmp.inertia_})

c_df = pl.DataFrame(results)
print("Effect of MSM cost parameter c:")
c_df

In [ ]:
c_df.hvplot.line(
    x="c",
    y="silhouette",
    width=700,
    height=350,
    title="Silhouette Score vs MSM Cost c",
    logx=True,
    markers=True,
)

## 14.5 Comparing with DTW k-means

`TimeSeriesKMeans` uses DTW + DBA (Dynamic Barycenter Averaging) for centroid
updates. Let's compare it side-by-side with KASBA on the same data.

In [ ]:
# KASBA (MSM)
kasba_labels = kasba(df, k=3, seed=42)
kasba_sil = silhouette_score(df, kasba_labels, method="msm")

# DTW k-means (DBA)
dba_clf = TimeSeriesKMeans(n_clusters=3, metric="dtw", max_iter=10, seed=42)
dba_clf.fit(df)
dba_labels = dba_clf.labels_
dba_sil = silhouette_score(df, dba_labels, method="dtw")

print(f"KASBA (MSM):      silhouette = {kasba_sil:.4f}")
print(f"DTW k-means (DBA): silhouette = {dba_sil:.4f}")

In [ ]:
# Compare cluster purity for both methods
for name, lbl_df in [("KASBA", kasba_labels), ("DTW k-means", dba_labels)]:
    merged = lbl_df.join(gt_df, on="unique_id")
    total_pure = 0
    for cid in merged["cluster"].unique().sort().to_list():
        members = merged.filter(pl.col("cluster") == cid)
        majority = members["true_cluster"].value_counts().sort("count", descending=True)["count"][0]
        total_pure += majority
    purity = total_pure / merged.height
    print(f"{name:15s}: purity = {purity:.0%}")

## 14.6 Stochastic barycenters — visualize learned centroids

KASBA learns centroids via stochastic barycenter averaging (SGD on a random subset
of cluster members). Let's visualize them and compare with DBA centroids from
DTW k-means.

In [ ]:
# Extract KASBA centroids
kasba_clf = KASBAClusterer(n_clusters=3, seed=42)
kasba_clf.fit(df)

centroid_rows = []
for i in range(3):
    centroid = kasba_clf.centroids_[i]  # shape: (1 * n_timepoints,)
    for t, v in enumerate(centroid):
        centroid_rows.append({"centroid": f"KASBA_{i}", "ds": t, "y": v})

# Extract DBA centroids
for i in range(3):
    centroid = dba_clf.centroids_[i]  # shape: (n_timepoints,)
    for t, v in enumerate(centroid):
        centroid_rows.append({"centroid": f"DBA_{i}", "ds": t, "y": v})

centroids_df = pl.DataFrame(centroid_rows)

centroids_df.hvplot.line(
    x="ds",
    y="y",
    by="centroid",
    width=900,
    height=400,
    title="Learned Centroids: KASBA (MSM) vs DBA (DTW)",
    line_width=2,
)

## 14.7 Predicting new series

The OOP API (`KASBAClusterer`) supports `predict()` — assign new, unseen series
to existing clusters without re-fitting.

In [ ]:
# Create 3 new series — one from each family
new_rows = []
rng_new = np.random.default_rng(777)

# New sine
vals = np.sin(np.linspace(0, 4 * np.pi, n_timepoints)) + rng_new.normal(0, 0.1, n_timepoints)
for t, v in enumerate(vals):
    new_rows.append({"unique_id": "new_sine", "ds": t, "y": v})

# New sawtooth
vals = np.linspace(0, 1, n_timepoints) + rng_new.normal(0, 0.05, n_timepoints)
for t, v in enumerate(vals):
    new_rows.append({"unique_id": "new_saw", "ds": t, "y": v})

# New step
vals = np.concatenate([np.zeros(25), np.ones(25)]) + rng_new.normal(0, 0.05, n_timepoints)
for t, v in enumerate(vals):
    new_rows.append({"unique_id": "new_step", "ds": t, "y": v})

new_df = pl.DataFrame(new_rows)
predictions = kasba_clf.predict(new_df)
print("Predictions for new series:")
predictions

## Summary

| Feature | API |
|---------|-----|
| Quick clustering | `kasba(df, k=3)` |
| OOP with fit/predict | `KASBAClusterer(n_clusters=3).fit(df)` |
| Multivariate | `kasba(df, k=3, channel_col="channel")` |
| Tune MSM cost | `kasba(df, k=3, c=2.0)` |
| Independent vs dependent | `KASBAClusterer(independent=False)` |

### When to use KASBA vs other methods

- **KASBA** — best when you need a proper metric (triangle inequality enables pruning),
  or when amplitude differences matter (tunable via `c`).
- **DTW k-means** — well-understood default; use when you want standard elastic alignment.
- **K-Shape** — fastest option; use when shape matters but phase/amplitude don't.
- **HDBSCAN** — use when you don't know `k` and want automatic noise detection.

### Key parameters

- `c` — MSM cost. Start with 1.0, sweep [0.1, 10.0] to tune.
- `independent` — for multivariate data. True (default) sums per-channel distances;
  False computes cross-channel MSM.
- `ba_subset_size` — fraction of cluster used for barycenter SGD. Lower = faster but noisier.
- `max_iter` — convergence typically within 5-10 iterations.